In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from torch.utils.data import WeightedRandomSampler
from fastai.callback.tracker import SaveModelCallback, Recorder
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import random_tfm, BlurPadDataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.show import (
    get_preds_for_ds,
    show_classification_report,
    show_confusion_matrix,
    show_confusion_matrix_using_preds,
)
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from functools import partial
from sklearn.model_selection import train_test_split
from fastai.basics import DataLoaders, default_device
from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain
from fastai.callback.all import ProgressCallback
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner, xresnet18

In [ ]:
from mtrain.utils import globL, mkdir

# CLEAN_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train"
# )
# FOVEATED_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated"
# )
FOVEATED_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/dataset"
)
# UNIT_TEST_DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/unit-tests/negmask")
# mkdir(UNIT_TEST_DEST / "train")
# mkdir(UNIT_TEST_DEST / "masks")

# images = globL(CLEAN_PATH / "train", "*.jpg")

# import shutil
# images = images[:256]
# for img in images:
#     mask = CLEAN_PATH / "masks" / f"{img.stem}.png"

#     shutil.copy(img, UNIT_TEST_DEST / "train" / img.name)
#     shutil.copy(mask, UNIT_TEST_DEST / "masks" / mask.name)

DS_PATH = FOVEATED_DS_PATH

In [ ]:
def get_learner(dls):
    learn = vision_learner(
        dls,
        xresnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=3,
        pretrained=True,
    )
    learn.remove_cb(Recorder)
    learn.remove_cb(SaveModelCallback)
    learn.add_cb(Recorder())
    learn.add_cbs([SaveModelCallback(monitor="f1_score", fname="best")])
    learn = learn.remove_cb(ProgressCallback)
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens[:3])
    image = image.permute([1, 2, 0]).numpy()
    mask = tens[3]
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPad4ChanDataset
CLS_WEIGHT = torch.tensor([1.0, 1.3]).float().to("mps")


def get_weight(p, taco_weight=1.0, manual_weight=1.2, default_weight=1.0):
    p = Path(p)
    if "taco" in p.name:
        return taco_weight
    if "manual" in p.name:
        return manual_weight
    else:
        return default_weight


def get_dls(
    num_samples,
    tfm,
    crop_size=224,
    ds_path=DS_PATH,
    min_area=35,
    min_bbox_length=3,
    max_area=None,
    path_filter=None,
):

    image_paths = list((ds_path / "train").glob("*.jpg"))[:num_samples]
    if path_filter is not None:
        image_paths = list(filter(path_filter, image_paths))
    stratify = [BlurPad4ChanDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )

    train_ds = BlurPad4ChanDataset(
        train_paths,
        ds_path / "masks",
        crop_size,
        False,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    valid_ds = BlurPad4ChanDataset(
        valid_paths,
        ds_path / "masks",
        crop_size,
        True,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    image_paths = train_ds.image_paths
    train_weights = [get_weight(p) for p in image_paths]
    train_sampler = WeightedRandomSampler(
        weights=train_weights, num_samples=len(image_paths), replacement=True
    )
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
        persistent_workers=True,
        dl_kwargs=[
            {"sampler": train_sampler, "shuffle": False},
            {"shuffle": False},  # Validation defaults
        ],
    )  # don't respawn workers each epoch)
    return dls


def vis_sample_ds(ds, idx):
    tens, targ = ds[idx]
    print("target", targ)
    print("shape", tens.shape)
    img, _ = get_denormalized(tens)
    plt.imshow(img, cmap="gray")
    plt.show()


def vis_sample(dls, idx):
    vis_sample_ds(dls.train_ds, idx)

In [ ]:
# to counter the problem of the model focusing on texture/noise
# we decrease the probability of adding noise with each sweep while maintaining accuracy
# the next step is to remove overwrite noise
# then next is decreasing the add noise frequency
# first i would need to seee the performance of the model
#  on different types of aux transforms (step down? gaussian? blur?)
# our final model has no noise, and one kind of step down function
# we need to test it on all transforms and find the winner
# for each we do successive training by decreasing the add_noise chance parameter
def blur_tfm(
    cropped_image, mask, inner_bbox, add_noise_chance, blur_kernel_sz, blur_sigma
):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_blur(blur_kernel_sz, blur_sigma)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop


def step_down_tfm(cropped_image, mask, inner_bbox, add_noise_chance, ratio):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_gauss_tfm(cropped_image, mask, inner_bbox, add_noise_chance, min_value):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(min_value)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop

In [ ]:
def get_initialised_learner():
    dls = get_dls(100, random_tfm)
    learner = get_learner(dls)
    MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")
    path = MODELS_DIR / "foveated-224" / "iter-7-xresnet18.pth"
    state_dict = torch.load(path)
    learner.model.load_state_dict(state_dict)
    return learner

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)

In [ ]:
from mtrain.utils import show_negmask_ds

show_negmask_ds(DS_PATH, 4)

In [ ]:
# def no_mapi_walls(path):
#     is_wall_mapi = "walls-mapillary" in Path(path).stem
#     return not is_wall_mapi


st_ed_tfm0 = partial(st_ed_tfm, add_noise_chance=-1)
dls = get_dls(100, st_ed_tfm0)

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class CustomN(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.body = encoder
        self.head = head
        self.mask_weight = nn.Parameter(torch.full((1, 512, 1, 1), 0.5))

    def forward(self, x):
        images = x[:,:3,:,:]
        masks = x[:,3:4,:,:]
        activs = self.body(images)
        pooled = F.adaptive_max_pool2d(masks, (7, 7))
        m = (pooled * (1.0 - self.mask_weight)) + self.mask_weight
        constrained_activs = activs * m
        out = self.head(constrained_activs)
        return out

In [ ]:
def rm_mapi_walls(path):
    if "mapillary" in path.name:
        return False
    else:
        return True
dls = get_dls(20000, st_ed_tfm0, path_filter=rm_mapi_walls)

In [ ]:
learn = get_initialised_learner()
body = learn.model[0]
head = learn.model[1]


model = CustomN(body, head)
learn.model = model
learn.dls = dls
learn.n_in = 4
learn.splitter = lambda m: [m.body, m.head]

In [ ]:
# sd = torch.load("/Users/hariomnarang/Desktop/personal/roads/datasets/models/check-focus-lin-layers/custom-mask-mult.pt")
# learn.model.load_state_dict(sd, strict=False)

In [ ]:
learn.model.state_dict().keys()

In [ ]:
learn.model.eval()
with torch.no_grad():
    _ = show_reports(learn)

In [ ]:
learn.freeze()
learn.fit_one_cycle(4)

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(5)

In [ ]:
_ = show_reports(learn)

In [ ]:
torch.save(learn.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/check-focus-lin-layers/custom-mask-mult-with-learnable-param.pt")